# Colab QLoRA Fine-Tuning Notebook — 5B-Class Model

This notebook fine-tunes a **5B-class instruction model** using **QLoRA** on Google Colab.

Default model:

```python
Qwen/Qwen3-4B-Instruct-2507
```

Why this model?

- It is close to the requested 5B scale.
- It is practical for Colab with 4-bit QLoRA.
- It is easier to run than jumping directly to 7B/14B on limited GPU memory.

You can switch to a heavier model later by changing `MODEL_ID`.

> Important: this notebook is for testing the fine-tuning pipeline. For real quality improvement, use a clean dataset with at least a few hundred high-quality examples and a separate eval set.


## 0. Colab setup checklist

Before running:

1. Open **Runtime → Change runtime type**.
2. Select **GPU**.
3. Prefer **T4 / L4 / A100**.
4. Run cells from top to bottom.
5. Start with a small dataset and `MAX_SEQ_LENGTH = 512` or `1024`.

If you get CUDA OOM, reduce:

```python
MAX_SEQ_LENGTH
per_device_train_batch_size
lora_r
gradient_accumulation_steps
```


In [ ]:
# Check GPU
!nvidia-smi

Fri May  1 15:27:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Install dependencies
# Restart runtime if Colab asks after installation.

!pip -q install -U \
  transformers \
  accelerate \
  peft \
  bitsandbytes \
  datasets \
  sentencepiece \
  protobuf \
  huggingface_hub

In [ ]:
import os
import json
import torch
import random
import numpy as np

from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print("GPU capability:", major, minor)


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU capability: 7 5


## 1. Optional Hugging Face login

For the default Qwen model, login is usually not required for downloading.

Use login if:

- your model is gated,
- you want to push the adapter to Hugging Face Hub,
- you want to download private models/datasets.

In Colab, you can store `HF_TOKEN` in **Secrets** and enable notebook access.


In [ ]:
# Optional login
# In Colab: left sidebar → key icon → add HF_TOKEN

try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face.")
    else:
        print("No HF_TOKEN found. Continuing without login.")
except Exception as e:
    print("Skipping HF login:", e)


Skipping HF login: Secret HF_TOKEN does not exist.


## 2. Model configuration

Default: `Qwen/Qwen3-4B-Instruct-2507`.

This is a **4B model**, but it is a good practical substitute for a 5B-class Colab test.

If you want to go heavier later:

```python
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
```

But 7B may be slower or may OOM on a free T4 GPU.


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# Colab-safe defaults
MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = "/content/qwen_5b_class_qlora_output"
ADAPTER_DIR = "/content/qwen_5b_class_qlora_adapter"

SYSTEM_PROMPT = """You are a ServiceNow API assistant.
Convert the user's request into a safe structured API action.
Return only valid JSON.
Do not execute the action.
Do not invent sys_id values.
If a sys_id is required but not provided, first create a lookup action."""


## 3. Create or load dataset

Expected JSONL format for your own data:

```json
{"messages":[{"role":"system","content":"..."},{"role":"user","content":"..."},{"role":"assistant","content":"..."}]}
```

The final assistant message is treated as the training target.  
The system/user messages are treated as prompt context and are masked from the loss.


In [ ]:
# Demo dataset
# Replace this with your own JSONL for real training.
# This sample is intentionally small. It is only for pipeline testing.

SAMPLE_DATA = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Create a high priority incident for email server down. Caller is amit@example.com. Short description should be Email server outage."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "create_incident",
                "method": "POST",
                "endpoint": "/api/now/table/incident",
                "query_params": {},
                "body": {
                    "caller_id": "amit@example.com",
                    "short_description": "Email server outage",
                    "description": "Email server is down.",
                    "priority": "2",
                    "urgency": "1",
                    "impact": "2"
                },
                "requires_lookup": False,
                "safe_to_execute": True
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Assign incident INC0012345 to Network team."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "lookup_assignment_group",
                "method": "GET",
                "endpoint": "/api/now/table/sys_user_group",
                "query_params": {
                    "sysparm_query": "name=Network",
                    "sysparm_fields": "sys_id,name",
                    "sysparm_limit": "1"
                },
                "body": {},
                "next_action_after_lookup": {
                    "action": "update_incident_assignment_group",
                    "method": "PATCH",
                    "endpoint": "/api/now/table/incident/{incident_sys_id}",
                    "body_template": {
                        "assignment_group": "{group_sys_id}"
                    }
                },
                "requires_lookup": True,
                "safe_to_execute": False
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Close incident INC0099887 with resolution notes: restarted mail service and verified delivery."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "lookup_incident",
                "method": "GET",
                "endpoint": "/api/now/table/incident",
                "query_params": {
                    "sysparm_query": "number=INC0099887",
                    "sysparm_fields": "sys_id,number,state",
                    "sysparm_limit": "1"
                },
                "body": {},
                "next_action_after_lookup": {
                    "action": "close_incident",
                    "method": "PATCH",
                    "endpoint": "/api/now/table/incident/{incident_sys_id}",
                    "body_template": {
                        "state": "7",
                        "close_notes": "restarted mail service and verified delivery",
                        "close_code": "Solved (Permanently)"
                    }
                },
                "requires_lookup": True,
                "safe_to_execute": False
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Find open incidents assigned to Priya."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "lookup_user",
                "method": "GET",
                "endpoint": "/api/now/table/sys_user",
                "query_params": {
                    "sysparm_query": "name=Priya",
                    "sysparm_fields": "sys_id,name,email",
                    "sysparm_limit": "5"
                },
                "body": {},
                "next_action_after_lookup": {
                    "action": "list_open_incidents_assigned_to_user",
                    "method": "GET",
                    "endpoint": "/api/now/table/incident",
                    "query_params_template": {
                        "sysparm_query": "assigned_to={user_sys_id}^active=true",
                        "sysparm_fields": "number,short_description,state,priority,assigned_to",
                        "sysparm_limit": "20"
                    }
                },
                "requires_lookup": True,
                "safe_to_execute": False
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Create a normal priority incident for laptop not charging. Caller is neha@example.com."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "create_incident",
                "method": "POST",
                "endpoint": "/api/now/table/incident",
                "query_params": {},
                "body": {
                    "caller_id": "neha@example.com",
                    "short_description": "Laptop not charging",
                    "description": "User reports that the laptop is not charging.",
                    "priority": "3",
                    "urgency": "2",
                    "impact": "2"
                },
                "requires_lookup": False,
                "safe_to_execute": True
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Delete incident INC0011223 immediately."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "refuse_unsafe_delete",
                "method": "NONE",
                "endpoint": "NONE",
                "query_params": {},
                "body": {
                    "reason": "Deleting an incident is destructive and requires explicit confirmation and a validated sys_id. No delete action will be generated."
                },
                "requires_lookup": False,
                "safe_to_execute": False
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Update INC0007777 priority to critical."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "lookup_incident",
                "method": "GET",
                "endpoint": "/api/now/table/incident",
                "query_params": {
                    "sysparm_query": "number=INC0007777",
                    "sysparm_fields": "sys_id,number,priority",
                    "sysparm_limit": "1"
                },
                "body": {},
                "next_action_after_lookup": {
                    "action": "update_incident_priority",
                    "method": "PATCH",
                    "endpoint": "/api/now/table/incident/{incident_sys_id}",
                    "body_template": {
                        "priority": "1",
                        "urgency": "1",
                        "impact": "1"
                    }
                },
                "requires_lookup": True,
                "safe_to_execute": False
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "List top 10 critical active incidents."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "list_critical_active_incidents",
                "method": "GET",
                "endpoint": "/api/now/table/incident",
                "query_params": {
                    "sysparm_query": "active=true^priority=1",
                    "sysparm_fields": "number,short_description,state,priority,assigned_to,opened_at",
                    "sysparm_limit": "10",
                    "sysparm_order_by": "opened_at"
                },
                "body": {},
                "requires_lookup": False,
                "safe_to_execute": True
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Create a change request for database patching this Saturday from 1 AM to 3 AM."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Change Request Table API",
                "action": "create_change_request",
                "method": "POST",
                "endpoint": "/api/now/table/change_request",
                "query_params": {},
                "body": {
                    "short_description": "Database patching",
                    "description": "Database patching planned this Saturday from 1 AM to 3 AM.",
                    "type": "normal",
                    "risk": "3",
                    "impact": "3"
                },
                "requires_lookup": False,
                "safe_to_execute": True,
                "notes": "The exact date should be resolved by the calling application before execution."
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Get details of user rahul@example.com."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow User Table API",
                "action": "get_user_details",
                "method": "GET",
                "endpoint": "/api/now/table/sys_user",
                "query_params": {
                    "sysparm_query": "email=rahul@example.com",
                    "sysparm_fields": "sys_id,name,email,user_name,department,active",
                    "sysparm_limit": "1"
                },
                "body": {},
                "requires_lookup": False,
                "safe_to_execute": True
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Create catalog request for new monitor for rohan@example.com."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Service Catalog API",
                "action": "lookup_catalog_item",
                "method": "GET",
                "endpoint": "/api/now/table/sc_cat_item",
                "query_params": {
                    "sysparm_query": "nameLIKEmonitor^active=true",
                    "sysparm_fields": "sys_id,name",
                    "sysparm_limit": "5"
                },
                "body": {},
                "next_action_after_lookup": {
                    "action": "create_catalog_request",
                    "method": "POST",
                    "endpoint": "/api/sn_sc/servicecatalog/items/{catalog_item_sys_id}/order_now",
                    "body_template": {
                        "sysparm_quantity": "1",
                        "variables": {
                            "requested_for": "rohan@example.com"
                        }
                    }
                },
                "requires_lookup": True,
                "safe_to_execute": False
            }, indent=2)}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Reopen INC0055555 because the issue has returned."},
            {"role": "assistant", "content": json.dumps({
                "servicenow_reference": "ServiceNow Incident Table API",
                "action": "lookup_incident",
                "method": "GET",
                "endpoint": "/api/now/table/incident",
                "query_params": {
                    "sysparm_query": "number=INC0055555",
                    "sysparm_fields": "sys_id,number,state",
                    "sysparm_limit": "1"
                },
                "body": {},
                "next_action_after_lookup": {
                    "action": "reopen_incident",
                    "method": "PATCH",
                    "endpoint": "/api/now/table/incident/{incident_sys_id}",
                    "body_template": {
                        "state": "2",
                        "work_notes": "Issue has returned. Reopening incident."
                    }
                },
                "requires_lookup": True,
                "safe_to_execute": False
            }, indent=2)}
        ]
    }
]

dataset = Dataset.from_list(SAMPLE_DATA)
print(dataset)
print(dataset[0]["messages"])


Dataset({
    features: ['messages'],
    num_rows: 12
})
[{'role': 'system', 'content': "You are a ServiceNow API assistant.\nConvert the user's request into a safe structured API action.\nReturn only valid JSON.\nDo not execute the action.\nDo not invent sys_id values.\nIf a sys_id is required but not provided, first create a lookup action."}, {'role': 'user', 'content': 'Create a high priority incident for email server down. Caller is amit@example.com. Short description should be Email server outage.'}, {'role': 'assistant', 'content': '{\n  "servicenow_reference": "ServiceNow Incident Table API",\n  "action": "create_incident",\n  "method": "POST",\n  "endpoint": "/api/now/table/incident",\n  "query_params": {},\n  "body": {\n    "caller_id": "amit@example.com",\n    "short_description": "Email server outage",\n    "description": "Email server is down.",\n    "priority": "2",\n    "urgency": "1",\n    "impact": "2"\n  },\n  "requires_lookup": false,\n  "safe_to_execute": true\n}'}]

### Optional: Upload your own JSONL dataset

Uncomment and run this cell if you want to upload your own `train.jsonl`.

Expected one record per line:

```json
{"messages":[{"role":"system","content":"..."},{"role":"user","content":"..."},{"role":"assistant","content":"..."}]}
```


In [ ]:
# Optional upload dataset
# from google.colab import files
# uploaded = files.upload()
# file_name = list(uploaded.keys())[0]
# dataset = load_dataset("json", data_files=file_name, split="train")
# print(dataset)
# print(dataset[0])


## 4. Load tokenizer and model in 4-bit

We use:

- 4-bit NF4 quantization
- double quantization
- LoRA adapters
- gradient checkpointing

This is QLoRA-style training.


In [ ]:
def supports_bf16():
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability()
    return major >= 8

compute_dtype = torch.bfloat16 if supports_bf16() else torch.float16
print("Compute dtype:", compute_dtype)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)

model.config.use_cache = False
print("Model loaded.")


Compute dtype: torch.float16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded.


## 5. Format and tokenize dataset

This notebook masks the loss for prompt tokens.

That means the model is mainly trained to predict the **assistant answer**, not to memorize the user prompt.


In [ ]:
def build_prompt_and_answer(messages):
    if len(messages) < 2:
        raise ValueError("Each example must contain at least user and assistant messages.")

    if messages[-1]["role"] != "assistant":
        raise ValueError("The final message must be from assistant because it is used as the target.")

    prompt_messages = messages[:-1]
    answer = messages[-1]["content"]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Add EOS so the model learns where to stop.
    answer_text = answer + tokenizer.eos_token

    return prompt_text, answer_text


def tokenize_example(example):
    prompt_text, answer_text = build_prompt_and_answer(example["messages"])
    full_text = prompt_text + answer_text

    tokenized_full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

    tokenized_prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]

    prompt_len = len(tokenized_prompt["input_ids"])
    prompt_len = min(prompt_len, len(input_ids))

    labels = [-100] * prompt_len + input_ids[prompt_len:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset.column_names,
)

split = tokenized_dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(train_dataset)
print(eval_dataset)
print("First tokenized length:", len(train_dataset[0]["input_ids"]))


Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 9
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})
First tokenized length: 191


In [ ]:
def data_collator(features):
    input_features = [
        {
            "input_ids": f["input_ids"],
            "attention_mask": f["attention_mask"],
        }
        for f in features
    ]

    batch = tokenizer.pad(
        input_features,
        padding=True,
        return_tensors="pt",
    )

    max_len = batch["input_ids"].shape[1]
    labels = []

    for f in features:
        label = f["labels"]
        pad_len = max_len - len(label)
        labels.append(label + [-100] * pad_len)

    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch


## 6. Add LoRA adapter

For Qwen-style transformer models, these target modules are commonly used:

```python
q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
```

Start with `r=16` or `r=32`.  
Increase only after your pipeline works.


In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


## 7. Test before fine-tuning

This gives you a baseline response before training.


In [ ]:
def generate_response(user_prompt, max_new_tokens=300):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    answer = decoded[len(text):]
    return answer.strip()


test_prompt = "Create a high priority incident for VPN not working. Caller is raj@example.com."
print(generate_response(test_prompt))


{
  "action": "create_incident",
  "priority": "1",
  "short_description": "VPN not working",
  "caller": "raj@example.com",
  "incident_type": "Technical Issue",
  "category": "Network",
  "subcategory": "VPN"
}<|im_end|>


## 8. Train

For a real run, increase dataset size first.  
Do not blindly increase epochs on tiny data; it will overfit.


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=supports_bf16(),
    fp16=not supports_bf16(),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,1.413280,0.950285


TrainOutput(global_step=2, training_loss=1.642558991909027, metrics={'train_runtime': 21.5234, 'train_samples_per_second': 0.418, 'train_steps_per_second': 0.093, 'total_flos': 47562383861760.0, 'train_loss': 1.642558991909027, 'epoch': 1.0})

## 9. Test after fine-tuning

Compare this with the earlier output.


In [ ]:
print(generate_response(test_prompt))

extra_tests = [
    "List top 10 critical active incidents.",
    "Update INC0001111 priority to critical.",
    "Delete incident INC0004444 now.",
    "Create catalog request for laptop for maya@example.com.",
]

for prompt in extra_tests:
    print("\nUSER:", prompt)
    print("ASSISTANT:")
    print(generate_response(prompt))
    print("-" * 80)


{
  "apiVersion": "v1",
  "action": "create_incident",
  "parameters": {
    "short_description": "VPN not working",
    "priority": "1",
    "caller": "raj@example.com",
    "u_cause": "VPN connectivity issue",
    "u_impact": "User unable to access network via VPN"
  },
  "required_fields": [
    "short_description",
    "priority",
    "caller"
  ],
  "validation": {
    "caller": {
      "type": "email",
      "required": true
    },
    "priority": {
      "type": "enum",
      "enum_values": [
        "1",
        "2",
        "3",
        "4",
        "5"
      ],
      "required": true
    }
  }
}<|im_end|>

USER: List top 10 critical active incidents.
ASSISTANT:
{
  "action": "query",
  "query": "SELECT sys_id, short_description, priority, state, created_date FROM incident WHERE state = 'Active' AND priority IN ('Critical', 'High') ORDER BY created_date DESC LIMIT 10",
  "table": "incident",
  "fields": [
    "sys_id",
    "short_description",
    "priority",
    "state",
    

## 10. Save adapter

This saves only the LoRA adapter, not the full base model.

That is what you usually want.


In [ ]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter to:", ADAPTER_DIR)

!du -sh /content/qwen_5b_class_qlora_adapter


Saved adapter to: /content/qwen_5b_class_qlora_adapter
137M	/content/qwen_5b_class_qlora_adapter


In [ ]:
# Zip and download adapter
!zip -r /content/qwen_5b_class_qlora_adapter.zip /content/qwen_5b_class_qlora_adapter

try:
    from google.colab import files
    files.download("/content/qwen_5b_class_qlora_adapter.zip")
except Exception as e:
    print("Download skipped:", e)
    print("Adapter zip path: /content/qwen_5b_class_qlora_adapter.zip")


  adding: content/qwen_5b_class_qlora_adapter/ (stored 0%)
  adding: content/qwen_5b_class_qlora_adapter/chat_template.jinja (deflated 73%)
  adding: content/qwen_5b_class_qlora_adapter/adapter_config.json (deflated 59%)
  adding: content/qwen_5b_class_qlora_adapter/adapter_model.safetensors (deflated 25%)
  adding: content/qwen_5b_class_qlora_adapter/tokenizer.json (deflated 81%)
  adding: content/qwen_5b_class_qlora_adapter/tokenizer_config.json (deflated 60%)
  adding: content/qwen_5b_class_qlora_adapter/README.md (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Load adapter later

Use this when you want to reload your fine-tuned adapter for inference.


In [ ]:
# Reload adapter later

# from peft import PeftModel

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
#     torch_dtype=compute_dtype,
# )

# fine_tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
# fine_tuned_model.eval()


## 12. Practical tuning knobs

If output is poor:

### Data problems

- Add more examples.
- Use consistent JSON keys.
- Include negative/refusal examples.
- Add lookup-required examples.
- Add edge cases.

### Training problems

Try:

```python
num_train_epochs = 2
learning_rate = 1e-4
lora_r = 32
MAX_SEQ_LENGTH = 1024
```

### OOM problems

Try:

```python
MAX_SEQ_LENGTH = 512
lora_r = 8
gradient_accumulation_steps = 4
```

### Production suggestion

Do not rely only on fine-tuning for ServiceNow knowledge.

Use:

```text
Fine-tuned model = structured API planner
ServiceNow API/tools = real execution
Lookup layer = fetch sys_id safely
Validator = checks generated JSON schema
```
